In [2]:
import json
import polars as pl

def load_data(file_path):
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

def create_dataframe():

    data = load_data('/Users/jameslee/Downloads/assignment1/Data_Set.json')
        
    destinations = data['destinations']
    df_destinations = pl.DataFrame({
        'destination_id': [d['destination_id'] for d in destinations],
        'longitude': [d['location']['longitude'] for d in destinations],
        'latitude': [d['location']['latitude'] for d in destinations]
    })
        
    orders = data['orders']
    df_orders = pl.DataFrame({
        'order_number': [o['order_number'] for o in orders],
        'box_id': [o['box_id'] for o in orders],
        'destination': [o['destination'] for o in orders],
        'width': [o['dimension']['width'] for o in orders],
        'length': [o['dimension']['length'] for o in orders],
        'height': [o['dimension']['height'] for o in orders]
    })

    depot = data['depot']
    depot_row = pl.DataFrame({
    'Vehicle_ID': pl.Series([0], dtype=pl.Int32),
    'Route_Order': pl.Series([0], dtype=pl.Int32),
    'Destination': 'Depot',
    'Order_Number': pl.Series([None], dtype=pl.Int64),
    'Box_ID': pl.Series([None], dtype=pl.String),
    'Stacking_Order': pl.Series([None], dtype=pl.Int32),
    'Lower_Left_X': pl.Series([None], dtype=pl.Int32),
    'Lower_Left_Y': pl.Series([None], dtype=pl.Int32),
    'Lower_Left_Z': pl.Series([None], dtype=pl.Int32),
    'Longitude': depot['location']['longitude'], 
    'Latitude': depot['location']['latitude']
    })

    depot_row = depot_row.with_columns([
    pl.lit(depot['dimension']['width']).cast(pl.Int64).alias('Box_Width'),
    pl.lit(depot['dimension']['length']).cast(pl.Int64).alias('Box_Length'),
    pl.lit(depot['dimension']['height']).cast(pl.Int64).alias('Box_Height')
])

    result = df_orders.join(
            df_destinations,
            left_on='destination',
            right_on='destination_id',
            how='left'
        )
    result = result.with_columns([
        pl.lit(0).alias('Vehicle_ID'),
        pl.lit(0).alias('Route_Order'),
        pl.lit(0).alias('Stacking_Order'),
        pl.lit(0).alias('Lower_Left_X'),
        pl.lit(0).alias('Lower_Left_Y'),
        pl.lit(0).alias('Lower_Left_Z'),
    ])

    result = result.select([
            'Vehicle_ID',
            'Route_Order',
            pl.col('destination').alias('Destination'),
            pl.col('order_number').alias('Order_Number'),
            pl.col('box_id').alias('Box_ID'),
            'Stacking_Order',
            'Lower_Left_X',
            'Lower_Left_Y',
            'Lower_Left_Z',
            pl.col('longitude').alias('Longitude'),
            pl.col('latitude').alias('Latitude'),
            pl.col('width').alias('Box_Width'),
            pl.col('length').alias('Box_Length'),
            pl.col('height').alias('Box_Height'),
        ])
        
    result = pl.concat([depot_row, result,depot_row])
    return result

df = create_dataframe()

In [3]:
df.head()

Vehicle_ID,Route_Order,Destination,Order_Number,Box_ID,Stacking_Order,Lower_Left_X,Lower_Left_Y,Lower_Left_Z,Longitude,Latitude,Box_Width,Box_Length,Box_Height
i32,i32,str,i64,str,i32,i32,i32,i32,f64,f64,i64,i64,i64
0,0,"""Depot""",null,null,null,null,null,null,129.075087,35.17982,0,0,0
0,0,"""D_00001""",1,"""B_00001""",0,0,0,0,129.033916,35.149045,30,40,30
0,0,"""D_00002""",2,"""B_00002""",0,0,0,0,129.071372,35.159031,50,60,50
0,0,"""D_00003""",3,"""B_00003""",0,0,0,0,129.026275,35.152292,50,60,50
0,0,"""D_00003""",4,"""B_00004""",0,0,0,0,129.026275,35.152292,50,60,50


In [4]:
df.tail()

Vehicle_ID,Route_Order,Destination,Order_Number,Box_ID,Stacking_Order,Lower_Left_X,Lower_Left_Y,Lower_Left_Z,Longitude,Latitude,Box_Width,Box_Length,Box_Height
i32,i32,str,i64,str,i32,i32,i32,i32,f64,f64,i64,i64,i64
0,0,"""D_00298""",434,"""B_00434""",0,0,0,0,129.088084,35.186753,30,50,40
0,0,"""D_00299""",435,"""B_00435""",0,0,0,0,129.09405,35.174711,30,50,40
0,0,"""D_00300""",436,"""B_00436""",0,0,0,0,129.095908,35.187242,30,50,40
0,0,"""D_00300""",437,"""B_00437""",0,0,0,0,129.095908,35.187242,30,40,30
0,0,"""Depot""",null,null,null,null,null,null,129.075087,35.17982,0,0,0


In [6]:
matrix = pl.read_csv(
    "/Users/jameslee/Downloads/assignment1/distance-data.txt", 
    separator="\t", 
    has_header=True, # first row is header (column names)
)

In [7]:
matrix

ORIGIN,DESTINATION,TIME_MIN,DISTANCE_METER
str,str,f64,i64
"""D_00139""","""D_00117""",24.179,13466
"""D_00139""","""D_00088""",31.67,22688
"""D_00139""","""D_00214""",32.196,18964
"""D_00139""","""D_00267""",36.986,24358
"""D_00139""","""D_00269""",34.992,27725
…,…,…,…
"""D_00009""","""D_00055""",9.645,3141
"""D_00009""","""Depot""",5.525,2311
"""D_00009""","""D_00223""",16.329,6030
